# 15.9 序列推荐 / Sequential Recommendation (GRU4Rec · SASRec · BERT4Rec)

**中文**：前面所有模型都把用户的历史当作一个**无序集合**——只关心"你喜欢哪些物品"，不关心"你按什么顺序看的"。但很多场景里**顺序至关重要**：你刚买了手机，下一步大概率想要手机壳，而不是再来一台手机。**序列推荐**专门建模用户行为的**时间顺序**，预测**下一个**物品。本节直接复用 Part 12（RNN/Transformer）的内功，实现三大经典模型。
**English**: Every model so far treated user history as an **unordered set** — caring only "which items you liked," not "in what order." But order is often decisive: just bought a phone → you probably want a case next, not another phone. **Sequential recommendation** models the **temporal order** of user behavior to predict the **next** item. This section reuses our Part 12 (RNN/Transformer) skills to build three classics.

---

**中文**：三大模型，本节都从零实现：
**English**: Three models, all implemented from scratch:

- **GRU4Rec（2016）**：用 **GRU**（循环网络）顺序读入物品序列，用最后的隐藏状态预测下一个。会话推荐的开山之作。
- **SASRec（2018）**：**单向（因果）自注意力**——像 GPT 一样，每个位置只能看左边，自回归地预测下一个物品。序列推荐进入 Transformer 时代。
- **BERT4Rec（2019）**：**双向自注意力 + 完形填空（Cloze）**——像 BERT 一样随机 mask 掉序列中的物品，用左右双向上下文预测被 mask 的物品。

- **GRU4Rec (2016)**: a **GRU** reads the item sequence in order; the final hidden state predicts the next item. The pioneer of session-based recommendation.
- **SASRec (2018)**: **unidirectional (causal) self-attention** — like GPT, each position sees only the left, autoregressively predicting the next item. Sequential rec enters the Transformer era.
- **BERT4Rec (2019)**: **bidirectional self-attention + Cloze** — like BERT, randomly mask items and predict them from both-side context.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 高频）**
> **中文**：序列推荐 = 建模行为**顺序**、预测**下一个**。**GRU4Rec** 循环、**SASRec** 因果自注意力(自回归)、**BERT4Rec** 双向+Cloze。SASRec vs BERT4Rec ≈ GPT vs BERT 的关系：单向自回归 vs 双向完形填空。评估用 **留一法(leave-one-out)** + HR@K / NDCG@K（用最后一个交互当测试目标）。**关键认知**：序列模型只有在数据**真的有顺序信号**时才赢——能用"打乱顺序"对照实验来检验（打乱后掉点说明真的用了顺序）。**负采样评估会高估指标**，全量排序才诚实。
> **English**: Sequential rec = model behavior **order**, predict the **next**. **GRU4Rec** recurrent, **SASRec** causal self-attention (autoregressive), **BERT4Rec** bidirectional + Cloze. SASRec vs BERT4Rec ≈ GPT vs BERT: unidirectional autoregressive vs bidirectional cloze. Evaluate with **leave-one-out** + HR@K / NDCG@K (the last interaction is the target). **Key insight**: sequential models win only when the data truly has order signal — verify with a **shuffle control** (a drop after shuffling means order was actually used). **Sampled-negative eval inflates metrics**; full ranking is honest.


## 实验一：合成序列数据——证明"顺序建模"有用 / Synthetic: order modeling works

**中文**：先用一个**顺序起决定作用**的合成任务，证明这些架构确实在利用顺序。我们用一个"环形马尔可夫链"生成序列：下一个物品 = 当前物品 +1（环绕），有 90% 概率成立，10% 随机跳。于是**要预测下一个，就必须知道'最后一个是什么'**——纯靠流行度（所有物品出现频率几乎一样）完全没用，打乱顺序也会失效。
**English**: First a task where **order is decisive**, to prove these architectures truly use order. We generate sequences from a "ring Markov chain": next item = current + 1 (wrap), with 90% probability, else a random jump. So **predicting the next item requires knowing the last one** — pure popularity (all items roughly equally frequent) is useless, and shuffling breaks it.


In [ ]:

# ============================================================
# 合成环形马尔可夫序列 + 三个模型的零件 / synthetic ring-Markov sequences + model parts
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as Fnn, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)

def gen_ring(V=60, L=20, Nu=4000, p=0.9):
    seqs=[]
    for _ in range(Nu):
        cur=np.random.randint(1,V+1); s=[cur]
        for _ in range(L):
            cur = (cur % V)+1 if np.random.rand()<p else np.random.randint(1,V+1)   # 90% 走环 / ring step
            s.append(cur)
        seqs.append(s)
    return seqs, V+1
Sseqs, Sn = gen_ring()
SL=20
Str=[s[:-1] for s in Sseqs]; Stgt=[s[-1] for s in Sseqs]   # 留一：最后一个当目标 / leave-one-out
def rpad(s,Lp): return s+[0]*(Lp-len(s)) if len(s)<Lp else s[-Lp:]
SX=torch.tensor([rpad(s,SL) for s in Str]); SLEN=torch.tensor([min(len(s),SL) for s in Str])
Spop=np.bincount([i for s in Str for i in s],minlength=Sn)

def gather_last(h,LENb): return h[torch.arange(len(h)),LENb-1]     # 取最后一个真实位置 / last real pos

class GRU4Rec(nn.Module):
    def __init__(s,n,d=64): super().__init__(); s.emb=nn.Embedding(n,d,padding_idx=0); s.gru=nn.GRU(d,d,batch_first=True)
    def seqout(s,x): h,_=s.gru(s.emb(x)); return h               # (B,L,d) 每步隐状态 / per-step states
    def score(s,h): return h@s.emb.weight.T                       # 共享 embedding 打分 / tied softmax

class SASRec(nn.Module):
    def __init__(s,n,L,d=64,nh=2,nl=2):
        super().__init__(); s.L=L; s.emb=nn.Embedding(n,d,padding_idx=0); s.pos=nn.Embedding(L,d)
        lyr=nn.TransformerEncoderLayer(d,nh,d*2,0.1,batch_first=True); s.enc=nn.TransformerEncoder(lyr,nl)
    def seqout(s,x):
        pos=torch.arange(s.L).unsqueeze(0).expand(x.size(0),-1)
        h=s.emb(x)+s.pos(pos)
        cmask=torch.triu(torch.ones(s.L,s.L),1).bool()            # 因果掩码：只看左边 / causal mask
        return s.enc(h,mask=cmask,src_key_padding_mask=(x==0))
    def score(s,h): return h@s.emb.weight.T

def train_ar(model, X, ep=8, bs=256, lr=2e-3):
    """自回归训练：每个位置预测下一个物品 / autoregressive: predict next item at each position."""
    n=model.emb.weight.size(0); opt=torch.optim.Adam(model.parameters(),lr)
    for e in range(ep):
        for b in torch.randperm(len(X)).split(bs):
            xb=X[b]; tgt=torch.zeros_like(xb); tgt[:,:-1]=xb[:,1:]      # 目标=右移一位 / shifted target
            logits=model.score(model.seqout(xb))
            loss=Fnn.cross_entropy(logits.reshape(-1,n),tgt.reshape(-1),ignore_index=0)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def eval_next(model, X, LEN, seqs, tgt, K=10):
    """留一法：取最后真实位置的输出预测目标，全量排序算 HR@K / leave-one-out HR@K, full ranking."""
    model.eval(); hr=ndcg=0; c=0
    with torch.no_grad():
        for b in range(0,len(X),512):
            xb=X[b:b+512]; last=gather_last(model.seqout(xb),LEN[b:b+512]); sc=model.score(last); sc[:,0]=-1e9
            for k in range(len(xb)):
                t=tgt[b+k]; rank=(sc[k]>sc[k][t]).sum().item()             # 目标的排名 / rank of target
                if rank<K: hr+=1; ndcg+=1/np.log2(rank+2)
                c+=1
    return hr/c, ndcg/c

def pop_hr(seqs, tgt, pop, K=10):
    order=np.argsort(pop)[::-1][:K]; return np.mean([tgt[k] in order for k in range(len(seqs))])

syn={}
syn["GRU4Rec"]=eval_next(train_ar(GRU4Rec(Sn),SX), SX, SLEN, Str, Stgt)
syn["SASRec"] =eval_next(train_ar(SASRec(Sn,SL),SX), SX, SLEN, Str, Stgt)
syn_pop=pop_hr(Str,Stgt,Spop)
print(f"{'model':<12}{'HR@10':>9}{'NDCG@10':>10}")
for k in ["GRU4Rec","SASRec"]: print(f"{k:<12}{syn[k][0]:>9.4f}{syn[k][1]:>10.4f}")
print(f"{'Popularity':<12}{syn_pop:>9.4f}")


**中文**：现在实现 **BERT4Rec**：和前两者的**自回归**不同，它用**双向**注意力 + **完形填空（Cloze）**训练——随机把序列里一些物品替换成特殊的 `[MASK]` 标记，让模型用**左右两侧**上下文预测被遮住的物品。预测"下一个"时，就在序列末尾补一个 `[MASK]`，预测它。
**English**: Now **BERT4Rec**: unlike the autoregressive two above, it uses **bidirectional** attention + **Cloze** training — randomly replace some items with a special `[MASK]` token and predict them from **both-side** context. To predict "the next" item, append a `[MASK]` at the sequence end and predict it.


In [ ]:

# ============================================================
# BERT4Rec：双向 + Cloze 完形填空 / bidirectional + masked-item (Cloze) training
# ============================================================
class BERT4Rec(nn.Module):
    def __init__(s,n,L,d=64,nh=2,nl=2):
        super().__init__(); s.L=L; s.MASK=n          # 额外的 [MASK] token id = n / extra mask token
        s.emb=nn.Embedding(n+1,d,padding_idx=0); s.pos=nn.Embedding(L,d)
        lyr=nn.TransformerEncoderLayer(d,nh,d*2,0.1,batch_first=True); s.enc=nn.TransformerEncoder(lyr,nl)
        s.out=nn.Linear(d,n)                          # 预测原始物品(不含 mask) / predict real items
    def seqout(s,x):
        pos=torch.arange(s.L).unsqueeze(0).expand(x.size(0),-1)
        h=s.emb(x)+s.pos(pos)
        return s.enc(h, src_key_padding_mask=(x==0))  # 无因果掩码=双向 / no causal mask = bidirectional

def train_bert(model, X, LEN, ep=10, bs=256, lr=2e-3, pmask=0.2):
    n=model.out.out_features; opt=torch.optim.Adam(model.parameters(),lr)
    for e in range(ep):
        for b in torch.randperm(len(X)).split(bs):
            xb=X[b].clone(); real=(xb>0)
            mask=(torch.rand_like(xb,dtype=torch.float)<pmask)&real      # 随机选要遮的真实位置 / mask positions
            tgt=torch.where(mask, xb, torch.zeros_like(xb))               # 目标=被遮物品 / targets at masked
            xb[mask]=model.MASK                                           # 替换为 [MASK] / replace
            logits=model.out(model.seqout(xb))
            loss=Fnn.cross_entropy(logits.reshape(-1,n),tgt.reshape(-1),ignore_index=0)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def eval_bert(model, X, LEN, tgt, K=10):
    model.eval(); hr=ndcg=0; c=0
    with torch.no_grad():
        for b in range(0,len(X),512):
            xb=X[b:b+512].clone(); Lb=LEN[b:b+512]
            xb[torch.arange(len(xb)),Lb]=model.MASK     # 末尾补 [MASK] 预测下一个 / append [MASK]
            logits=model.out(model.seqout(xb))
            sc=logits[torch.arange(len(xb)),Lb]; sc[:,0]=-1e9
            for k in range(len(xb)):
                t=tgt[b+k]; rank=(sc[k]>sc[k][t]).sum().item()
                if rank<K: hr+=1; ndcg+=1/np.log2(rank+2)
                c+=1
    return hr/c, ndcg/c

# 末尾要留一个空位放 [MASK]，所以截断到 SL-1 再右补 / leave one slot for the appended MASK
SXb=torch.tensor([rpad(s,SL) for s in [t[:SL-1] for t in Str]])
SLENb=torch.tensor([min(len(s),SL-1) for s in Str])
syn["BERT4Rec"]=eval_bert(train_bert(BERT4Rec(Sn,SL),SXb,SLENb), SXb, SLENb, Stgt)
print(f"BERT4Rec  HR@10={syn['BERT4Rec'][0]:.4f}  NDCG@10={syn['BERT4Rec'][1]:.4f}")
print("\n合成任务总结 / synthetic summary (order is decisive):")
for k in ["GRU4Rec","SASRec","BERT4Rec"]: print(f"  {k:<10} HR@10={syn[k][0]:.4f}")
print(f"  {'Popularity':<10} HR@10={syn_pop:.4f}")


**中文**：三个模型都**远超流行度基线（≈0.16）**，证明它们确实在利用顺序。其中 **GRU4Rec / SASRec ≈ 0.9**，而 **BERT4Rec ≈ 0.38** 明显偏低——这是个诚实且重要的点：BERT4Rec 的**完形填空(Cloze)+双向**目标是为"恢复序列中间被遮的物品"设计的，与"严格预测**下一个**"存在**训练/评估失配**（预测下一个时右侧没有上下文可用）。所以在纯 next-item 任务上，**自回归的 GRU4Rec/SASRec 天然更合适**；BERT4Rec 的优势要在能利用双向上下文的场景才体现。下面进入真实数据，看看 MovieLens 是不是也这么"听话"。
**English**: All three **far exceed the popularity baseline (≈0.16)**, proving they do exploit order. But **GRU4Rec / SASRec ≈ 0.9** while **BERT4Rec ≈ 0.38** is clearly lower — an honest and important point: BERT4Rec's **Cloze + bidirectional** objective is built to "recover masked items inside a sequence," which has a **train/eval mismatch** with "strictly predict the **next**" (no right-side context exists when predicting the next). So on pure next-item tasks, **autoregressive GRU4Rec/SASRec fit naturally better**; BERT4Rec's edge appears where bidirectional context is usable. Now to real data — is MovieLens as cooperative?


In [ ]:

# ============================================================
# 真实数据：MovieLens 用户观影序列(按时间) / real MovieLens viewing sequences (chronological)
# ============================================================
import os, pandas as pd
R=os.path.expanduser("~/.cache/dsfs_recsys/ml-100k")
rat=pd.read_csv(os.path.join(R,"u.data"),sep="\t",names=["user","item","rating","ts"])
iids=np.sort(rat["item"].unique()); i2x={i:j+1 for j,i in enumerate(iids)}; Rn=len(iids)+1
Rseqs=[]
for _,g in rat.sort_values("ts").groupby("user"):
    s=[i2x[i] for i in g["item"]]
    if len(s)>=5: Rseqs.append(s)
RL=50
Rtr=[s[:-1][-RL:] for s in Rseqs]; Rtgt=[s[-1] for s in Rseqs]
RX=torch.tensor([rpad(s,RL) for s in Rtr]); RLEN=torch.tensor([min(len(s),RL) for s in Rtr])
Rpop=np.bincount([i for s in Rtr for i in s],minlength=Rn)
print(f"序列数 / sequences: {len(Rseqs)}, 物品 items: {Rn-1}, 平均长度 avg len: {np.mean([len(s) for s in Rtr]):.0f}")

# 训练 SASRec / train SASRec on real data
sas_real=train_ar(SASRec(Rn,RL), RX, ep=15)
hr_o,nd_o=eval_next(sas_real, RX, RLEN, Rtr, Rtgt)
# GRU4Rec 对照 / GRU4Rec
gru_real=train_ar(GRU4Rec(Rn), RX, ep=15)
hr_g,nd_g=eval_next(gru_real, RX, RLEN, Rtr, Rtgt)
hr_pop=pop_hr(Rtr,Rtgt,Rpop)
print(f"\n{'model':<14}{'HR@10':>9}{'NDCG@10':>10}")
print(f"{'SASRec':<14}{hr_o:>9.4f}{nd_o:>10.4f}")
print(f"{'GRU4Rec':<14}{hr_g:>9.4f}{nd_g:>10.4f}")
print(f"{'Popularity':<14}{hr_pop:>9.4f}")


**中文**：诚实结果——在 MovieLens-100k 上，**序列模型并没有打败流行度基线**（HR@10 都很低）。为什么差这么多？我们用一个干净的**对照实验**给出答案：把每个用户的序列**随机打乱**再训练 SASRec。如果顺序真的重要，打乱后应该明显掉点。
**English**: Honest result — on MovieLens-100k, **the sequential models do not beat the popularity baseline** (low HR@10 across the board). Why so different from the synthetic? A clean **control experiment** answers it: randomly **shuffle** each user's sequence and retrain SASRec. If order truly matters, shuffling should cause a clear drop.


In [ ]:

# ============================================================
# 对照实验：打乱序列顺序 / control: shuffle the order, retrain
# ============================================================
import random
Rtr_shuf=[s[:] for s in Rtr]
for s in Rtr_shuf: random.Random(0).shuffle(s)
RXs=torch.tensor([rpad(s,RL) for s in Rtr_shuf])
sas_shuf=train_ar(SASRec(Rn,RL), RXs, ep=15)
hr_s,nd_s=eval_next(sas_shuf, RXs, RLEN, Rtr_shuf, Rtgt)
print(f"SASRec 原始顺序 / ordered : HR@10={hr_o:.4f}")
print(f"SASRec 打乱顺序 / shuffled: HR@10={hr_s:.4f}")
print(f"差异 / difference: {hr_o-hr_s:+.4f}  ->", "顺序几乎无用 / order barely matters" if abs(hr_o-hr_s)<0.01 else "顺序有用 / order matters")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(12,4.3))
# ① 合成任务：序列模型 vs 热门 / synthetic: sequential models crush popularity
names=["GRU4Rec","SASRec","BERT4Rec","Popularity"]; vals=[syn["GRU4Rec"][0],syn["SASRec"][0],syn["BERT4Rec"][0],syn_pop]
ax[0].bar(names,vals,color=["#4C72B0","#55A868","#8172B3","#C44E52"])
for i,v in enumerate(vals): ax[0].text(i,v+0.02,f"{v:.2f}",ha="center")
ax[0].set_title("合成(顺序决定)：序列模型完胜 / synthetic: order decisive"); ax[0].set_ylabel("HR@10"); ax[0].set_ylim(0,1)
# ② 真实任务：原始 vs 打乱 vs 热门 / real: ordered vs shuffled vs popularity
n2=["SASRec\n原始/ordered","SASRec\n打乱/shuffled","Popularity"]; v2=[hr_o,hr_s,hr_pop]
ax[1].bar(n2,v2,color=["#55A868","#937860","#C44E52"])
for i,v in enumerate(v2): ax[1].text(i,v+0.002,f"{v:.3f}",ha="center")
ax[1].set_title("真实 MovieLens：打乱几乎不掉点→弱序列 / weakly sequential"); ax[1].set_ylabel("HR@10")
plt.tight_layout(); plt.savefig("/tmp/rec09_viz.png",dpi=80); plt.show()
print("结论：合成数据顺序决定一切，真实 MovieLens 顺序信号很弱")


**中文**：把两个实验合起来，得到本节最重要的诚实洞察：
**English**: Combining both experiments yields this section's key honest insight:

**中文**：
1. **架构是有效的**：合成"环形马尔可夫"数据上，GRU4Rec / SASRec / BERT4Rec 的 HR@10 都 ≈ 0.9，远超流行度——只要顺序里有信号，它们就能抓住。
2. **但 MovieLens 几乎没有顺序信号**：打乱顺序后 SASRec 的 HR@10 **几乎不变**（差异 < 0.01）。这说明它根本没在用顺序，只是在学"这个用户喜欢的物品集合"的协同信号——而且因为数据太小（943 序列）、太稀疏，连流行度都打不过。
3. **这解释了一个常见误区**：很多人以为"上了 SASRec/Transformer 就一定更好"。真相是：**序列模型只在真正序列化的数据上（如电商加购→下单、视频连播、新闻流）才发挥威力**。MovieLens 是"评分快照"，不是"行为流"，所以序列建模收益甚微。**选模型前先验证数据是否有序列性**（打乱对照实验就是最简单的探针）。

**English**:
1. **The architectures work**: on the synthetic ring-Markov data, GRU4Rec / SASRec / BERT4Rec all reach HR@10 ≈ 0.9, far above popularity — given order signal, they capture it.
2. **But MovieLens has almost no order signal**: shuffling barely changes SASRec's HR@10 (difference < 0.01). It isn't using order at all — just learning the collaborative "set of items this user likes" — and with so little data (943 sequences) and sparsity, it can't even beat popularity.
3. **This debunks a common myth**: many assume "switching to SASRec/Transformer must be better." Truth: **sequential models shine only on genuinely sequential data** (e-commerce add-to-cart → purchase, video binge-watching, news feeds). MovieLens is a "ratings snapshot," not a "behavior stream," so sequential modeling barely helps. **Verify your data has sequential structure before choosing the model** — the shuffle control is the simplest probe.

> 💼 **实战视角 / Practical angle**
> **中文**：① 序列推荐是电商/短视频/feed 的主力（阿里 DIN/DIEN、SASRec、BERT4Rec 系）。② 评估务必用**全量排序 + 留一法**；论文里常见的"对 100 个负样本排序"会严重高估指标，面试要能指出。③ SASRec(单向)适合"预测下一个"，BERT4Rec(双向)在某些数据上更强但**不能直接做自回归生成**。④ 永远做**打乱对照**确认顺序确有价值。
> **English**: ① Sequential rec dominates e-commerce / short-video / feeds (Alibaba DIN/DIEN, SASRec, BERT4Rec family). ② Always evaluate with **full ranking + leave-one-out**; the common "rank against 100 negatives" massively inflates metrics — be ready to call this out in interviews. ③ SASRec (unidirectional) fits "predict next"; BERT4Rec (bidirectional) can be stronger on some data but **can't autoregressively generate**. ④ Always run the **shuffle control** to confirm order truly matters.

---
### 小结 / Summary
- **中文**：序列推荐建模行为顺序预测下一个；GRU4Rec 循环、SASRec 因果注意力、BERT4Rec 双向 Cloze。
- **English**: Sequential rec models behavior order to predict the next; GRU4Rec recurrent, SASRec causal attention, BERT4Rec bidirectional Cloze.
- **中文**：合成顺序数据上 GRU4Rec/SASRec≈0.9、BERT4Rec≈0.38（Cloze 与 next-item 失配）但都完胜热门；真实 MovieLens 打乱几乎不掉点=弱序列、收益甚微。
- **English**: On synthetic ordered data GRU4Rec/SASRec≈0.9, BERT4Rec≈0.38 (Cloze vs next-item mismatch) but all beat popularity; on real MovieLens shuffling barely hurts = weakly sequential, little gain.
- **中文**：用全量排序+留一法评估，用打乱对照验证顺序价值——别迷信"换 Transformer 就更好"。
- **English**: Evaluate with full-ranking leave-one-out; use the shuffle control to verify order's value — don't assume "Transformer = better."
